In [0]:
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%run ../../utils/reader_utils

In [0]:

%run ../../utils/writer_utils

In [0]:
%run ../../utils/transform_utils

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
today = date.today()

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', False)
START_DATE = configs.get("start_date") or today - timedelta(days=1)
END_DATE = configs.get("end_date") or today + timedelta(days=1)
CATALOG = f"sl_{ENV}"

print(ENV, INITIAL_RUN, START_DATE, END_DATE, CATALOG)

In [0]:
# Constants
SILVER_VEHICLE_TABLE_CONF = {
    "table": "pyspark_vehicle_telemetry",
    "schema": "silver",
    "timestamp_col": "_insert_update_ts"
}

DIM_VEHICLE_TABLE_CONF = {
    "table": "pyspark_dim_vehicle",
    "schema": "gold",
}

DIM_DATE_TABLE_CONF = {
    "table": "pyspark_dim_date",
    "schema": "gold",
}

TARGET_TABLE_CONF = {
    "table": "pyspark_fact_vehicle_telemetry",
    "schema": "gold",
    "merge_keys": ["vehicle_id", "period_start_timestamp"],
    "primary_keys": ["vehicle_id", "period_start_timestamp"]
}

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
silver_telemetry_df = (read_delta_table(SILVER_VEHICLE_TABLE_CONF, START_DATE, END_DATE)
                      .drop("_insert_update_ts")).filter(F.col("vehicle_id") == "VH-001").orderBy("reading_timestamp")

dim_vehicle_df = (spark.table(f"""{DIM_VEHICLE_TABLE_CONF.get("schema")}.
                        {DIM_VEHICLE_TABLE_CONF.get("table")}"""))
                        
dim_date_df = (spark.table(f"""{DIM_DATE_TABLE_CONF.get("schema")}.
                        {DIM_DATE_TABLE_CONF.get("table")}"""))

In [0]:
telemetry_periodic = (silver_telemetry_df
                      .withColumn("period_start_timestamp", F.date_trunc("hour", F.col("reading_timestamp")))
                      .groupBy("vehicle_id", F.col("period_start_timestamp"))
                      .agg(F.count(F.col("reading_id")).alias("readings_count"),
                           F.round(F.avg(F.col("speed_kmh"))).alias("avg_speed_kmh"),
                           F.max(F.col("speed_kmh")).alias("max_speed_kmh"),
                           F.min(F.col("odometer_km")).alias("min_odometer_km"),
                           F.max(F.col("odometer_km")).alias("max_odometer_km"),
                           F.round(F.avg(F.col("fuel_pct")), 2).alias("avg_fuel_pct"),
                           F.round(F.avg(F.col("engine_temp_c"))).alias("avg_engine_temp_c"),
                           F.min(F.col("cargo_temp_c")).alias("min_cargo_temp_c"),
                           F.max(F.col("cargo_temp_c")).alias("max_cargo_temp_c"),
                           F.sum(F.when(F.col("vehicle_status").isin("idle", "maintenance"), F.lit(0)).otherwise(F.lit(1))).alias("vehicle_idle"))
                      .withColumn("km_driven", F.col("max_odometer_km") - F.col("min_odometer_km"))
                      .withColumn("utilization_pct", F.round(F.col("vehicle_idle") / F.col("readings_count"), 2))
                      .withColumn("period_date", F.col("period_start_timestamp").cast(DateType()))
                      .drop("min_odometer_km", "max_odometer_km", "vehicle_idle", "_insert_update_ts")
                      )

In [0]:
fact_vehicle_telemetry_df = (telemetry_periodic.hint("skew", "vehicle_id").alias("main")
                             .join(dim_vehicle_df.alias("v"), "vehicle_id", "left")
                             .join(dim_date_df.alias("d"), F.col("main.period_date") == F.col("d.full_date"), "left")
                             .select("main.vehicle_id",
                                     "main.period_start_timestamp",
                                     "v.vehicle_sk",
                                     "d.date_key",
                                     *[F.col(f"main.{c}") for c in 
                                       telemetry_periodic.columns if c not in ["vehicle_id", "period_start_timestamp"]]))

In [0]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    target_table = f"{TARGET_TABLE_CONF.get('schema')}.{TARGET_TABLE_CONF.get('table')}"
    primary_keys = TARGET_TABLE_CONF.get('primary_keys')

    (fact_vehicle_telemetry_df
     .writeTo(target_table)
     .using("delta")
     .createOrReplace())
    
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY AUTO")

    for key in primary_keys:
        spark.sql(f"ALTER TABLE {target_table} ALTER COLUMN {key} SET NOT NULL")

    spark.sql(f"""ALTER TABLE {target_table} ADD CONSTRAINT vehicle_telemetry_pk PRIMARY KEY ({", ".join(primary_keys)})""")

    spark.sql(f"""ALTER TABLE {target_table} ADD CONSTRAINT fct_vhcl_vehicle_sk_fk FOREIGN KEY (vehicle_sk) 
              REFERENCES {DIM_VEHICLE_TABLE_CONF.get('schema')}.{DIM_VEHICLE_TABLE_CONF.get('table')}(vehicle_sk)""")

    spark.sql(f"""ALTER TABLE {target_table} ADD CONSTRAINT fct_vhcl_date_key_fk FOREIGN KEY (date_key) 
              REFERENCES {DIM_DATE_TABLE_CONF.get('schema')}.{DIM_DATE_TABLE_CONF.get('table')}(date_key)""")
    
    dbutils.notebook.exit(f"Initial Table: {target_table} Setup Finished Successfully")

In [0]:
upsert_to_delta(fact_vehicle_telemetry_df, TARGET_TABLE_CONF)